# Multi-Encoding Super-Ensemble for PTM Prediction (14+ Models)

This notebook creates a comprehensive **super-ensemble** combining:
- **12 ML models**: RF + XGBoost + SVM trained on 4 encodings each (onehot, blosum, aapc, hybrid)
- **2-3 DL models**: CNN + CRNN + Transformer+GRU trained on raw sequences

## Three-Branch Architecture

**Branch 1 - Chemical ML Pipeline:**
- Random Forest × 4 encodings = 4 models
- XGBoost × 4 encodings = 4 models
- LinearSVM × 4 encodings = 4 models
- **Total: 12 ML models**

**Branch 2 - Sequence DL Pipeline:**
- CNN (Convolutional Neural Network) = 1 model
- CRNN (Conv + BiGRU + Attention) = 1 model
- Transformer+GRU (ESM-2 + BiGRU) = 1 model
- **Total: 3 DL models**

**Branch 3 - Ensemble Manager:**
- Weighted soft voting with optimized weights
- Grid search to find optimal combination
- **Total: 14-15 models combined**

## Why Multi-Encoding + Multi-Architecture?

Different encodings capture different patterns:
- **One-Hot**: Position-specific amino acid identity
- **BLOSUM62**: Evolutionary similarity between amino acids
- **AAPC**: Local dipeptide patterns
- **Hybrid**: BLOSUM + physicochemical properties

Different architectures capture different patterns:
- **RF/XGBoost/SVM**: Non-linear feature interactions
- **CNN**: Local motifs and spatial patterns
- **CRNN**: Local + sequential dependencies
- **Transformer**: Pre-trained protein knowledge + sequential patterns

## Expected Performance

- Single best model: F1 = 0.25-0.35, AUC = 0.78-0.85
- Simple average ensemble: F1 = 0.35-0.42, AUC = 0.82-0.88
- **Optimized 14+ model ensemble**: F1 = 0.42-0.50, AUC = 0.85-0.92

Target improvement: **+60-80%** over best single model

## 1. Configuration

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    f1_score, 
    roc_auc_score, 
    hamming_loss,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

print("Libraries imported successfully")
print(f"Timestamp: {datetime.now().isoformat()}")

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Encodings used for ML models
ENCODINGS = ["onehot", "blosum", "aapc", "hybrid"]

# Label columns
label_cols = ["S-glutathionylation", "S-nitrosylation", "S-palmitoylation"]

# File paths
VAL_FILE = "../data/val.csv"  # Validation file with true labels

# ML model predictions (nested by encoding)
# Format: ../output/{model_type}/{encoding}/{model_type}_val_predictions.csv
ML_MODEL_TYPES = ["rf", "xgb", "svm"]

# DL model predictions
# Format: ../output/{model_type}/{run_name}/{model_type}_val_predictions.csv
DL_MODELS = {
    "cnn": "../output/cnn/run4/cnn_val_predictions.csv",
    "crnn": "../output/crnn/run1_baseline/crnn_val_predictions.csv",
    "transformer_gru": "../output/transformer_gru/run1_frozen_esm2/transformer_gru_val_predictions.csv"
}

# Output configuration
EXPERIMENT_NAME = "run1_14plus_models"
OUTPUT_DIR = f"../output/ensemble/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_WEIGHTS = os.path.join(OUTPUT_DIR, "multi_encoding_ensemble_weights.pkl")
OUTPUT_PREDICTIONS = os.path.join(OUTPUT_DIR, "multi_encoding_ensemble_val_predictions.csv")
OUTPUT_RESULTS = os.path.join(OUTPUT_DIR, "multi_encoding_ensemble_results.csv")
VISUALIZATIONS_DIR = os.path.join(OUTPUT_DIR, "visualizations")
os.makedirs(VISUALIZATIONS_DIR, exist_ok=True)

# Grid search configuration
GRID_SEARCH_SAMPLES = 5000  # Number of random weight combinations to test
RANDOM_STATE = 42

print("\n" + "="*70)
print("MULTI-ENCODING ENSEMBLE CONFIGURATION")
print("="*70)
print(f"Encodings: {ENCODINGS}")
print(f"ML model types: {ML_MODEL_TYPES}")
print(f"DL models: {list(DL_MODELS.keys())}")
print(f"Expected total models: {len(ML_MODEL_TYPES) * len(ENCODINGS) + len(DL_MODELS)}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Grid search samples: {GRID_SEARCH_SAMPLES:,}")
print("="*70)

## 2. Load Ground Truth Labels

In [ ]:
# Load validation data with true labels
val_df = pd.read_csv(VAL_FILE)

# Extract labels
y_true = val_df[label_cols].values

print(f"Validation samples: {len(val_df):,}")
print(f"Labels shape: {y_true.shape}")
print(f"\nClass distribution:")
for idx, label in enumerate(label_cols):
    pos_count = y_true[:, idx].sum()
    print(f"  {label}: {pos_count:,} positive ({pos_count/len(y_true)*100:.2f}%)")

## 3. Load Model Predictions

Load predictions from all available models:
- **ML models**: RF, XGBoost, SVM (4 encodings each = 12 models)
- **DL models**: CNN, CRNN, Transformer+GRU (3 models)

The notebook gracefully handles missing models.

In [ ]:
def load_ml_predictions(model_type, encoding):
    """
    Load ML model predictions from nested directory structure.
    
    Path format: ../output/{model_type}/{encoding}/{model_type}_val_predictions.csv
    Column format: {model_type}_glut_proba, {model_type}_nitro_proba, {model_type}_palm_proba
    
    Parameters:
    -----------
    model_type : str
        Model type (e.g., 'rf', 'xgb', 'svm')
    encoding : str
        Encoding type (e.g., 'onehot', 'blosum', 'aapc', 'hybrid')
    
    Returns:
    --------
    predictions : np.ndarray or None
        Shape (n_samples, 3) if successful, None if file not found
    """
    pred_file = f"../output/{model_type}/{encoding}/{model_type}_val_predictions.csv"
    
    if not os.path.exists(pred_file):
        return None
    
    try:
        df = pd.read_csv(pred_file)
        
        # Extract probability columns
        prob_cols = [
            f"{model_type}_glut_proba",
            f"{model_type}_nitro_proba",
            f"{model_type}_palm_proba"
        ]
        
        predictions = df[prob_cols].values
        return predictions
        
    except Exception as e:
        print(f"    ERROR loading {pred_file}: {e}")
        return None


def load_dl_predictions(model_name, file_path):
    """
    Load DL model predictions.
    
    Column format: {model_name}_glut_proba, {model_name}_nitro_proba, {model_name}_palm_proba
    
    Parameters:
    -----------
    model_name : str
        Model name (e.g., 'cnn', 'crnn', 'transformer_gru')
    file_path : str
        Full path to predictions file
    
    Returns:
    --------
    predictions : np.ndarray or None
        Shape (n_samples, 3) if successful, None if file not found
    """
    if not os.path.exists(file_path):
        return None
    
    try:
        df = pd.read_csv(file_path)
        
        # Extract probability columns
        prob_cols = [
            f"{model_name}_glut_proba",
            f"{model_name}_nitro_proba",
            f"{model_name}_palm_proba"
        ]
        
        predictions = df[prob_cols].values
        return predictions
        
    except Exception as e:
        print(f"    ERROR loading {file_path}: {e}")
        return None

In [ ]:
predictions = {}

print("\n" + "="*70)
print("LOADING MODEL PREDICTIONS")
print("="*70)

# Load ML model predictions (RF, XGBoost, SVM)
for model_type in ML_MODEL_TYPES:
    print(f"\n{model_type.upper()} models:")
    for encoding in ENCODINGS:
        pred = load_ml_predictions(model_type, encoding)
        if pred is not None:
            model_key = f"{model_type}_{encoding}"
            predictions[model_key] = pred
            print(f"  ✓ Loaded {model_key} (shape: {pred.shape})")
        else:
            print(f"  ✗ Missing {model_type}_{encoding}")

# Load DL model predictions
print("\nDeep Learning models:")
for model_name, file_path in DL_MODELS.items():
    pred = load_dl_predictions(model_name, file_path)
    if pred is not None:
        predictions[model_name] = pred
        print(f"  ✓ Loaded {model_name} (shape: {pred.shape})")
    else:
        print(f"  ✗ Missing {model_name} (optional)")

print(f"\n{'='*70}")
print(f"✓ Total models loaded: {len(predictions)}")
print(f"{'='*70}")

# Show loaded models by category
print("\nLoaded models by category:")
ml_models = [k for k in predictions.keys() if any(k.startswith(m) for m in ML_MODEL_TYPES)]
dl_models = [k for k in predictions.keys() if k not in ml_models]

print(f"\nML Models ({len(ml_models)}):")
for i, model_name in enumerate(sorted(ml_models), 1):
    print(f"  {i:2d}. {model_name}")

print(f"\nDL Models ({len(dl_models)}):")
for i, model_name in enumerate(sorted(dl_models), 1):
    print(f"  {i:2d}. {model_name}")

if len(predictions) == 0:
    raise ValueError("No model predictions found! Please train models first.")

## 4. Evaluate Individual Models

Before ensembling, evaluate each individual model to identify best performers.

In [ ]:
def evaluate_predictions(y_true, y_pred_proba, threshold=0.5):
    """
    Evaluate predictions and return comprehensive metrics.
    
    Parameters:
    -----------
    y_true : np.ndarray
        True labels, shape (n_samples, n_labels)
    y_pred_proba : np.ndarray
        Predicted probabilities, shape (n_samples, n_labels)
    threshold : float
        Classification threshold (default 0.5)
    
    Returns:
    --------
    macro_f1 : float
        Macro-averaged F1 score
    macro_auc : float
        Macro-averaged AUC-ROC
    results : dict
        Per-label results
    """
    y_pred = (y_pred_proba > threshold).astype(int)
    
    results = {}
    for idx, label in enumerate(label_cols):
        f1 = f1_score(y_true[:, idx], y_pred[:, idx], zero_division=0)
        auc = roc_auc_score(y_true[:, idx], y_pred_proba[:, idx])
        results[label] = {"f1": f1, "auc": auc}
    
    macro_f1 = np.mean([r["f1"] for r in results.values()])
    macro_auc = np.mean([r["auc"] for r in results.values()])
    
    return macro_f1, macro_auc, results

In [ ]:
print("\n" + "="*70)
print("INDIVIDUAL MODEL PERFORMANCE")
print("="*70)

individual_results = {}

print(f"\n{'Model':<30} {'Macro F1':<12} {'Macro AUC':<12}")
print("-" * 54)

for model_name, pred in sorted(predictions.items()):
    macro_f1, macro_auc, results = evaluate_predictions(y_true, pred)
    individual_results[model_name] = {"f1": macro_f1, "auc": macro_auc, "details": results}
    print(f"{model_name:<30} {macro_f1:<12.4f} {macro_auc:<12.4f}")

# Find best models
best_f1_model = max(individual_results.items(), key=lambda x: x[1]['f1'])
best_auc_model = max(individual_results.items(), key=lambda x: x[1]['auc'])

print(f"\n{'='*70}")
print(f"Best F1:  {best_f1_model[0]} ({best_f1_model[1]['f1']:.4f})")
print(f"Best AUC: {best_auc_model[0]} ({best_auc_model[1]['auc']:.4f})")
print(f"{'='*70}")

## 5. Strategy 1: Simple Average Ensemble

Baseline ensemble with equal weights for all models.

In [ ]:
print("\n" + "="*70)
print("STRATEGY 1: SIMPLE AVERAGE (EQUAL WEIGHTS)")
print("="*70)

# Average all model predictions with equal weights
pred_avg = np.mean(list(predictions.values()), axis=0)
f1_avg, auc_avg, results_avg = evaluate_predictions(y_true, pred_avg)

print(f"\n{'Label':<30} {'F1':<12} {'AUC':<12}")
print("-" * 54)
for label in label_cols:
    print(f"{label:<30} {results_avg[label]['f1']:<12.4f} {results_avg[label]['auc']:<12.4f}")

print(f"\n{'Macro Average':<30} {f1_avg:<12.4f} {auc_avg:<12.4f}")

# Calculate improvement over best single model
improvement_f1 = ((f1_avg - best_f1_model[1]['f1']) / best_f1_model[1]['f1']) * 100
improvement_auc = ((auc_avg - best_auc_model[1]['auc']) / best_auc_model[1]['auc']) * 100

print(f"\nImprovement over best single model:")
print(f"  F1:  {improvement_f1:+.2f}%")
print(f"  AUC: {improvement_auc:+.2f}%")

## 6. Strategy 2: Optimized Weights (Grid Search)

Find optimal weights for each model through randomized grid search.

**Approach**: With 14+ models, exhaustive search is infeasible. We use **Dirichlet sampling**:
- Sample random weight vectors that sum to 1.0
- Round to nearest 0.05 for stability
- Test 5000 combinations
- Select weights that maximize macro F1 score

In [ ]:
print("\n" + "="*70)
print("STRATEGY 2: OPTIMIZED WEIGHTS (GRID SEARCH)")
print("="*70)

model_names = list(predictions.keys())
n_models = len(model_names)

print(f"\nSearching optimal weights for {n_models} models...")
print(f"Method: Dirichlet sampling ({GRID_SEARCH_SAMPLES:,} iterations)")
print("This may take a few minutes...\n")

np.random.seed(RANDOM_STATE)

best_f1 = 0
best_weights = None
best_pred = None

# Random sampling using Dirichlet distribution
for iteration in range(GRID_SEARCH_SAMPLES):
    # Generate random weights that sum to 1.0
    weights = np.random.dirichlet(np.ones(n_models))
    
    # Round to nearest 0.05 for stability
    weights = np.round(weights * 20) / 20
    
    # Renormalize to ensure sum = 1.0
    weights = weights / weights.sum()
    
    # Calculate weighted predictions
    weighted_pred = sum(w * predictions[m] for w, m in zip(weights, model_names))
    
    # Evaluate
    f1, auc, _ = evaluate_predictions(y_true, weighted_pred)
    
    if f1 > best_f1:
        best_f1 = f1
        best_weights = {m: w for m, w in zip(model_names, weights)}
        best_pred = weighted_pred
    
    if (iteration + 1) % 1000 == 0:
        print(f"  Tested {iteration+1:,} combinations... (best F1 so far: {best_f1:.4f})")

print(f"\n✓ Search complete! Tested {GRID_SEARCH_SAMPLES:,} combinations.")
print(f"\nBest weights (F1={best_f1:.4f}):")
print("-" * 70)
for model, weight in sorted(best_weights.items(), key=lambda x: -x[1]):
    if weight > 0.01:  # Only show significant weights
        print(f"  {model:<30} {weight:.3f}")

## 7. Final Ensemble Evaluation

Comprehensive evaluation of the optimized ensemble.

In [ ]:
print("\n" + "="*70)
print("FINAL ENSEMBLE EVALUATION (OPTIMIZED WEIGHTS)")
print("="*70)

f1_opt, auc_opt, results_opt = evaluate_predictions(y_true, best_pred)

print(f"\n{'Label':<30} {'F1':<12} {'AUC':<12}")
print("-" * 54)
for label in label_cols:
    print(f"{label:<30} {results_opt[label]['f1']:<12.4f} {results_opt[label]['auc']:<12.4f}")

print(f"\n{'Macro Average':<30} {f1_opt:<12.4f} {auc_opt:<12.4f}")

# Calculate improvements
improvement_over_best_f1 = ((f1_opt - best_f1_model[1]['f1']) / best_f1_model[1]['f1']) * 100
improvement_over_best_auc = ((auc_opt - best_auc_model[1]['auc']) / best_auc_model[1]['auc']) * 100
improvement_over_avg_f1 = ((f1_opt - f1_avg) / f1_avg) * 100 if f1_avg > 0 else 0
improvement_over_avg_auc = ((auc_opt - auc_avg) / auc_avg) * 100 if auc_avg > 0 else 0

print(f"\n{'='*70}")
print("PERFORMANCE COMPARISON")
print("="*70)
print(f"\n{'Configuration':<30} {'Macro F1':<12} {'Macro AUC':<12}")
print("-" * 54)
print(f"{'Best single model':<30} {best_f1_model[1]['f1']:<12.4f} {best_auc_model[1]['auc']:<12.4f}")
print(f"{'Simple average ensemble':<30} {f1_avg:<12.4f} {auc_avg:<12.4f}")
print(f"{'Optimized weight ensemble':<30} {f1_opt:<12.4f} {auc_opt:<12.4f}")

print(f"\nImprovement over best single model:")
print(f"  F1:  {improvement_over_best_f1:+.2f}%")
print(f"  AUC: {improvement_over_best_auc:+.2f}%")

print(f"\nImprovement over simple average:")
print(f"  F1:  {improvement_over_avg_f1:+.2f}%")
print(f"  AUC: {improvement_over_avg_auc:+.2f}%")

## 8. Visualization: Individual Model Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, max(8, len(predictions) * 0.4)))

# Sort models by F1 score
sorted_models = sorted(individual_results.items(), key=lambda x: x[1]['f1'])
model_names_sorted = [m[0] for m in sorted_models]
f1_scores_sorted = [m[1]['f1'] for m in sorted_models]
auc_scores_sorted = [m[1]['auc'] for m in sorted_models]

# F1 scores
axes[0].barh(range(len(model_names_sorted)), f1_scores_sorted, color='skyblue', edgecolor='black')
axes[0].set_yticks(range(len(model_names_sorted)))
axes[0].set_yticklabels(model_names_sorted, fontsize=9)
axes[0].set_xlabel('Macro F1-score', fontsize=12)
axes[0].set_title('Individual Model Performance (F1)', fontsize=14, fontweight='bold')
axes[0].axvline(f1_opt, color='red', linestyle='--', linewidth=2, label=f'Optimized Ensemble: {f1_opt:.4f}')
axes[0].axvline(f1_avg, color='orange', linestyle=':', linewidth=2, label=f'Simple Average: {f1_avg:.4f}')
axes[0].legend(fontsize=10)
axes[0].grid(axis='x', alpha=0.3)

# AUC scores
axes[1].barh(range(len(model_names_sorted)), auc_scores_sorted, color='coral', edgecolor='black')
axes[1].set_yticks(range(len(model_names_sorted)))
axes[1].set_yticklabels(model_names_sorted, fontsize=9)
axes[1].set_xlabel('Macro AUC-ROC', fontsize=12)
axes[1].set_title('Individual Model Performance (AUC)', fontsize=14, fontweight='bold')
axes[1].axvline(auc_opt, color='red', linestyle='--', linewidth=2, label=f'Optimized Ensemble: {auc_opt:.4f}')
axes[1].axvline(auc_avg, color='orange', linestyle=':', linewidth=2, label=f'Simple Average: {auc_avg:.4f}')
axes[1].legend(fontsize=10)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'individual_model_performance.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved plot to {os.path.join(VISUALIZATIONS_DIR, 'individual_model_performance.png')}")

## 9. Visualization: Model Weights in Ensemble

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(6, len(best_weights) * 0.3)))

# Filter and sort weights (show only significant weights > 0.01)
significant_weights = {k: v for k, v in best_weights.items() if v > 0.01}
sorted_weights = sorted(significant_weights.items(), key=lambda x: -x[1])
weight_names = [w[0] for w in sorted_weights]
weight_values = [w[1] for w in sorted_weights]

# Color by model type
colors = []
for name in weight_names:
    if name.startswith('rf_'):
        colors.append('#3498db')  # Blue for RF
    elif name.startswith('xgb_'):
        colors.append('#e74c3c')  # Red for XGBoost
    elif name.startswith('svm_'):
        colors.append('#f39c12')  # Orange for SVM
    elif name == 'cnn':
        colors.append('#9b59b6')  # Purple for CNN
    elif name == 'crnn':
        colors.append('#1abc9c')  # Teal for CRNN
    elif name == 'transformer_gru':
        colors.append('#2ecc71')  # Green for Transformer
    else:
        colors.append('#95a5a6')  # Gray for others

ax.barh(range(len(weight_names)), weight_values, color=colors, edgecolor='black', linewidth=1.5)
ax.set_yticks(range(len(weight_names)))
ax.set_yticklabels(weight_names, fontsize=10)
ax.set_xlabel('Weight', fontsize=12)
ax.set_title(f'Optimized Model Weights in Final Ensemble\n(Macro F1 = {f1_opt:.4f})', 
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(weight_values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9, fontweight='bold')

# Add legend for model types
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', edgecolor='black', label='Random Forest'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='XGBoost'),
    Patch(facecolor='#f39c12', edgecolor='black', label='SVM'),
    Patch(facecolor='#9b59b6', edgecolor='black', label='CNN'),
    Patch(facecolor='#1abc9c', edgecolor='black', label='CRNN'),
    Patch(facecolor='#2ecc71', edgecolor='black', label='Transformer+GRU')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'ensemble_weights.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved plot to {os.path.join(VISUALIZATIONS_DIR, 'ensemble_weights.png')}")

## 10. Visualization: Per-Label Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Prepare data for plotting
best_single_f1 = [individual_results[best_f1_model[0]]['details'][label]['f1'] for label in label_cols]
avg_f1 = [results_avg[label]['f1'] for label in label_cols]
opt_f1 = [results_opt[label]['f1'] for label in label_cols]

best_single_auc = [individual_results[best_auc_model[0]]['details'][label]['auc'] for label in label_cols]
avg_auc = [results_avg[label]['auc'] for label in label_cols]
opt_auc = [results_opt[label]['auc'] for label in label_cols]

x = np.arange(len(label_cols))
width = 0.25

# F1 scores
axes[0].bar(x - width, best_single_f1, width, label='Best Single Model', color='skyblue', edgecolor='black')
axes[0].bar(x, avg_f1, width, label='Simple Average', color='lightcoral', edgecolor='black')
axes[0].bar(x + width, opt_f1, width, label='Optimized Ensemble', color='lightgreen', edgecolor='black')
axes[0].set_xlabel('PTM Label', fontsize=12)
axes[0].set_ylabel('F1 Score', fontsize=12)
axes[0].set_title('Per-Label F1 Score Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels([l.replace('S-', '') for l in label_cols], rotation=15, ha='right')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# AUC scores
axes[1].bar(x - width, best_single_auc, width, label='Best Single Model', color='skyblue', edgecolor='black')
axes[1].bar(x, avg_auc, width, label='Simple Average', color='lightcoral', edgecolor='black')
axes[1].bar(x + width, opt_auc, width, label='Optimized Ensemble', color='lightgreen', edgecolor='black')
axes[1].set_xlabel('PTM Label', fontsize=12)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title('Per-Label AUC Comparison', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([l.replace('S-', '') for l in label_cols], rotation=15, ha='right')
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'per_label_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved plot to {os.path.join(VISUALIZATIONS_DIR, 'per_label_comparison.png')}")

## 11. Save Ensemble Outputs

In [ ]:
print("\n" + "="*70)
print("SAVING ENSEMBLE OUTPUTS")
print("="*70)

# Save optimal weights
with open(OUTPUT_WEIGHTS, "wb") as f:
    pickle.dump(best_weights, f)
print(f"✓ Saved optimal weights to:\n  {OUTPUT_WEIGHTS}")

# Save ensemble predictions
pred_df = pd.DataFrame({
    "ID": val_df["ID"],
    "ensemble_glut_proba": best_pred[:, 0],
    "ensemble_nitro_proba": best_pred[:, 1],
    "ensemble_palm_proba": best_pred[:, 2]
})
pred_df.to_csv(OUTPUT_PREDICTIONS, index=False)
print(f"✓ Saved ensemble predictions to:\n  {OUTPUT_PREDICTIONS}")

# Save results summary
results_summary = pd.DataFrame({
    "configuration": ["best_single_model", "simple_average", "optimized_ensemble"],
    "model_name": [best_f1_model[0], "equal_weights", "grid_search_weights"],
    "n_models": [1, len(predictions), len([w for w in best_weights.values() if w > 0.01])],
    "macro_f1": [best_f1_model[1]['f1'], f1_avg, f1_opt],
    "macro_auc": [best_auc_model[1]['auc'], auc_avg, auc_opt],
    "improvement_f1_pct": [
        0.0,
        ((f1_avg - best_f1_model[1]['f1']) / best_f1_model[1]['f1']) * 100,
        improvement_over_best_f1
    ],
    "improvement_auc_pct": [
        0.0,
        ((auc_avg - best_auc_model[1]['auc']) / best_auc_model[1]['auc']) * 100,
        improvement_over_best_auc
    ]
})
results_summary.to_csv(OUTPUT_RESULTS, index=False)
print(f"✓ Saved results summary to:\n  {OUTPUT_RESULTS}")

# Save per-label results
per_label_results = []
for label in label_cols:
    per_label_results.append({
        "label": label,
        "best_single_f1": individual_results[best_f1_model[0]]['details'][label]['f1'],
        "best_single_auc": individual_results[best_auc_model[0]]['details'][label]['auc'],
        "simple_avg_f1": results_avg[label]['f1'],
        "simple_avg_auc": results_avg[label]['auc'],
        "optimized_f1": results_opt[label]['f1'],
        "optimized_auc": results_opt[label]['auc']
    })
per_label_df = pd.DataFrame(per_label_results)
per_label_path = os.path.join(OUTPUT_DIR, "per_label_results.csv")
per_label_df.to_csv(per_label_path, index=False)
print(f"✓ Saved per-label results to:\n  {per_label_path}")

print(f"\n✓ All visualizations saved to:\n  {VISUALIZATIONS_DIR}/")

## 12. Summary

In [ ]:
print("\n" + "="*70)
print("MULTI-ENCODING ENSEMBLE SUMMARY")
print("="*70)

print(f"\nModels Combined: {len(predictions)}")
print(f"  - ML Models: {len([k for k in predictions if any(k.startswith(m) for m in ML_MODEL_TYPES)])}")
print(f"  - DL Models: {len([k for k in predictions if k in DL_MODELS])}")

print(f"\nActive Models in Ensemble: {len([w for w in best_weights.values() if w > 0.01])}")
print(f"Grid Search Samples: {GRID_SEARCH_SAMPLES:,}")

print(f"\nFinal Performance:")
print(f"  Macro F1:  {f1_opt:.4f}")
print(f"  Macro AUC: {auc_opt:.4f}")

print(f"\nImprovement Over Best Single Model ({best_f1_model[0]}):")
print(f"  F1:  {improvement_over_best_f1:+.2f}%")
print(f"  AUC: {improvement_over_best_auc:+.2f}%")

print(f"\nPer-Label F1 Scores:")
for label in label_cols:
    print(f"  {label}: {results_opt[label]['f1']:.4f}")

print(f"\nOutput Directory: {OUTPUT_DIR}")
print(f"\nFiles Generated:")
print(f"  - multi_encoding_ensemble_weights.pkl")
print(f"  - multi_encoding_ensemble_val_predictions.csv")
print(f"  - multi_encoding_ensemble_results.csv")
print(f"  - per_label_results.csv")
print(f"  - visualizations/individual_model_performance.png")
print(f"  - visualizations/ensemble_weights.png")
print(f"  - visualizations/per_label_comparison.png")

print("\n" + "="*70)
print("✓ MULTI-ENCODING SUPER-ENSEMBLE COMPLETE!")
print("="*70)

print("\nNext Steps:")
print("1. Apply these ensemble weights to test set predictions")
print("2. Consider threshold optimization for further F1 improvement")
print("3. Analyze which models contribute most to ensemble performance")
print("4. Generate final submission file for competition/evaluation")
print("="*70)